In [8]:
from pxd.datasets.hdf5.directory import HDF5DirectoryDataset, HDF5DirectoryDatasetConfig
import numpy as np
import pandas as pd
from typing import Dict, Any, Tuple, List
import seaborn as sns
from tqdm import tqdm
from pathlib import Path
from samples.sample import Sample

In [5]:
config = HDF5DirectoryDatasetConfig(
    path="/flux/vault/stator/03_splitting/stator_splitting_ds2_nc/train"
)

dataset = HDF5DirectoryDataset(config)

In [6]:
dataset_dict = dataset.load()

samples = {}
for i in tqdm(list(dataset_dict.keys())):
    samples[i] = dataset_dict[i]()

  0%|          | 0/611 [00:00<?, ?it/s]

100%|██████████| 611/611 [00:24<00:00, 24.95it/s]


In [7]:
samples["sample_777"]

{'data': {'flags': {'high_a_matrix_error': array([False]),
   'high_q_error': array([False]),
   'missing_heatflux': array([False])},
  'heatflux_matrix': {'Q01': array([  123.82306636, -6003.67368739,  1577.29330223,  3018.1878916 ,
           4639.72037345, -4238.7803648 , -1701.45638982,  1602.39888116,
           1358.96284506, -4858.96340339,    63.08797136, -3273.95547755,
           -438.93992143,  5978.06607868, -5275.73602589, -1982.47826652]),
   'Q02': array([ 4250.44601466,   173.95390006,  1825.95958826,  1919.5648412 ,
          -2474.31981822, -7896.4445883 , -2379.82222598,  3780.53552455,
           5388.36490797,  5322.55081527, -5673.59026253, -2276.19286699,
          -1240.56492898, -2649.71058067,  4927.66951478,  7840.02663929]),
   'Q03': array([-4068.8414393 ,   595.79361894, -1077.11334321, -4036.88467374,
          -1228.89823163,  5609.42831337,  -918.54488328, -7290.53337013,
          -1179.5206075 ,  2126.94685254,  1420.1674406 ,  1238.01918178,
        

# Extract and Stack `heatflux_matrix` Data

This notebook extracts the following keys from each sample's `data -> heatflux_matrix` dictionary:

- `geometry_parameter_a_matrix`
- `a_matrix_coeffiecient_positions`
- `a_matrix_coefficients`

For each sample, the arrays are normalized to shapes `(120, 20)`, `(120, 2)`, and `(120, 1)`, then horizontally stacked into `(120, 23)`.
All samples are then vertically stacked into a final dataset of shape `(120 * number_of_samples, 23)`.

In [9]:

# ------------------------------
# Helper functions
# ------------------------------

def _ensure_2d(arr: Any) -> np.ndarray:
    """Convert input to a 2D numpy array."""
    arr = np.asarray(arr)
    if arr.ndim == 1:
        arr = arr.reshape(-1, 1)
    if arr.ndim != 2:
        raise ValueError(f"Expected a 2D array, got shape {arr.shape}")
    return arr


def _normalize_shape(arr: Any, expected_cols: int, rows_per_sample: int = 120, name: str = 'array') -> np.ndarray:
    """
    Normalize an input matrix so its final shape is (rows_per_sample, expected_cols).

    Accepts either:
      - (rows_per_sample, expected_cols)
      - (expected_cols, rows_per_sample)  -> transposed automatically
    """
    arr = _ensure_2d(arr)

    if arr.shape == (rows_per_sample, expected_cols):
        return arr
    elif arr.shape == (expected_cols, rows_per_sample):
        return arr.T
    else:
        raise ValueError(
            f"Unexpected shape for {name}: {arr.shape}. "
            f"Expected ({rows_per_sample}, {expected_cols}) or ({expected_cols}, {rows_per_sample})."
        )


# ------------------------------
# Main extraction function
# ------------------------------

def build_heatflux_dataset(
    samples_dict: Dict[str, Dict[str, Any]],
    rows_per_sample: int = 120
) -> Tuple[np.ndarray, Dict[str, np.ndarray], List[str]]:
    """
    Extracts and combines data from each sample:
      - geometry_parameter_a_matrix           -> 20 columns
      - a_matrix_coeffiecient_positions       -> 2 columns
      - a_matrix_coefficients                 -> 1 column

    For each sample, the three arrays are horizontally stacked into shape:
        (rows_per_sample, 23)

    Then all sample matrices are vertically stacked into shape:
        (rows_per_sample * number_of_samples, 23)

    Parameters
    ----------
    samples_dict : dict
        Expected structure:
        {
            sample_id: {
                "data": {
                    "heatflux_matrix": {
                        "geometry_parameter_a_matrix": ...,
                        "a_matrix_coeffiecient_positions": ...,
                        "a_matrix_coefficients": ...
                    },
                    ... # other dictionaries ignored
                }
            },
            ...
        }

    rows_per_sample : int, default=120
        Number of rows for each sample.

    Returns
    -------
    full_dataset : np.ndarray
        Shape = (rows_per_sample * n_samples, 23)

    per_sample_data : dict
        {sample_id: sample_matrix} where each sample_matrix has shape (rows_per_sample, 23)

    sample_order : list
        Order in which samples were processed and stacked.
    """
    per_sample_data: Dict[str, np.ndarray] = {}
    sample_order: List[str] = []

    for sample_id, sample_payload in samples_dict.items():
        try:
            heatflux = sample_payload["data"]["heatflux_matrix"]
        except KeyError as e:
            raise KeyError(f"Missing key for sample '{sample_id}': {e}") from e

        geom_a = _normalize_shape(
            heatflux["geometry_parameter_a_matrix"],
            expected_cols=20,
            rows_per_sample=rows_per_sample,
            name=f"{sample_id}.geometry_parameter_a_matrix"
        )

        coeff_pos = _normalize_shape(
            heatflux["a_matrix_coeffiecient_positions"],
            expected_cols=2,
            rows_per_sample=rows_per_sample,
            name=f"{sample_id}.a_matrix_coeffiecient_positions"
        )

        coeffs = _normalize_shape(
            heatflux["a_matrix_coefficients"],
            expected_cols=1,
            rows_per_sample=rows_per_sample,
            name=f"{sample_id}.a_matrix_coefficients"
        )

        # Horizontal stack within one sample -> (120, 23)
        sample_matrix = np.hstack([geom_a, coeff_pos, coeffs])

        if sample_matrix.shape != (rows_per_sample, 23):
            raise ValueError(
                f"Unexpected combined shape for sample '{sample_id}': {sample_matrix.shape}. "
                f"Expected ({rows_per_sample}, 23)."
            )

        per_sample_data[sample_id] = sample_matrix
        sample_order.append(sample_id)

    # Vertical stack across samples -> (120 * n_samples, 23)
    if not per_sample_data:
        full_dataset = np.empty((0, 23))
    else:
        full_dataset = np.vstack([per_sample_data[sid] for sid in sample_order])

    return full_dataset, per_sample_data, sample_order


In [11]:
full_dataset, _, sample_order = build_heatflux_dataset(samples)

print("Sample order:", sample_order)
'''for sid, arr in per_sample_data.items():
    print(f"{sid}: {arr.shape}")'''

print("Final dataset shape:", full_dataset.shape)  # Expected: (120 * number_of_samples, 23)

Sample order: ['sample_777', 'sample_615', 'sample_544', 'sample_969', 'sample_479', 'sample_936', 'sample_284', 'sample_122', 'sample_396', 'sample_152', 'sample_707', 'sample_665', 'sample_758', 'sample_456', 'sample_534', 'sample_919', 'sample_824', 'sample_381', 'sample_376', 'sample_685', 'sample_951', 'sample_441', 'sample_523', 'sample_710', 'sample_921', 'sample_553', 'sample_499', 'sample_602', 'sample_293', 'sample_306', 'sample_264', 'sample_761', 'sample_498', 'sample_920', 'sample_796', 'sample_842', 'sample_358', 'sample_265', 'sample_307', 'sample_292', 'sample_134', 'sample_328', 'sample_377', 'sample_144', 'sample_673', 'sample_711', 'sample_440', 'sample_684', 'sample_832', 'sample_153', 'sample_397', 'sample_202', 'sample_360', 'sample_825', 'sample_408', 'sample_918', 'sample_457', 'sample_759', 'sample_706', 'sample_937', 'sample_855', 'sample_478', 'sample_968', 'sample_545', 'sample_427', 'sample_729', 'sample_614', 'sample_776', 'sample_123', 'sample_285', 'samp

## convert final dataset to a pandas DataFrame

In [12]:
feature_names = [f"geometry_parameter_a_{i+1}" for i in range(20)] + [
    "a_matrix_coeff_position_1",
    "a_matrix_coeff_position_2",
    "a_matrix_coefficient"
]

df = pd.DataFrame(full_dataset, columns=feature_names)
df.head()


,geometry_parameter_a_1,geometry_parameter_a_2,geometry_parameter_a_3,geometry_parameter_a_4,geometry_parameter_a_5,geometry_parameter_a_6,geometry_parameter_a_7,geometry_parameter_a_8,geometry_parameter_a_9,geometry_parameter_a_10,...,geometry_parameter_a_14,geometry_parameter_a_15,geometry_parameter_a_16,geometry_parameter_a_17,geometry_parameter_a_18,geometry_parameter_a_19,geometry_parameter_a_20,a_matrix_coeff_position_1,a_matrix_coeff_position_2,a_matrix_coefficient
0,1.35128,1.623341,0.433823,0.070785,4.651849,2.259505,2.883141,1.792532,2.484936,0.310272,...,0.709467,0.603899,1.876252,1.627966,0.199283,0.143414,0.005288,0.0,1.0,1.761700
1,1.35128,1.623341,0.433823,0.070785,4.651849,2.259505,2.883141,1.792532,2.484936,0.310272,...,0.709467,0.603899,1.876252,1.627966,0.199283,0.143414,0.005288,0.0,2.0,19.790557
2,1.35128,1.623341,0.433823,0.070785,4.651849,2.259505,2.883141,1.792532,2.484936,0.310272,...,0.709467,0.603899,1.876252,1.627966,0.199283,0.143414,0.005288,0.0,3.0,28.758068
3,1.35128,1.623341,0.433823,0.070785,4.651849,2.259505,2.883141,1.792532,2.484936,0.310272,...,0.709467,0.603899,1.876252,1.627966,0.199283,0.143414,0.005288,0.0,4.0,0.018755
4,1.35128,1.623341,0.433823,0.070785,4.651849,2.259505,2.883141,1.792532,2.484936,0.310272,...,0.709467,0.603899,1.876252,1.627966,0.199283,0.143414,0.005288,0.0,5.0,0.000668


In [13]:
# Save as CSV if needed
df.to_csv("full_heatflux_dataset.csv", index=False)

print("Saved: full_heatflux_dataset.npy and full_heatflux_dataset.csv")

Saved: full_heatflux_dataset.npy and full_heatflux_dataset.csv
